# Hansard Preprocessing Demo

This notebook demonstrates the Hansard preprocessing pipeline:
1. **Parsing**: Extract speaker turns from raw text
2. **Validation**: Run T1-T10 quality checks
3. **Enrichment**: Match speakers to MP reference data

In [ ]:
# Setup
import sys
import os
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path('../src').resolve()))

from preprocessing import (
    HansardParser,
    validate_hansard,
    enrich_hansard,
    build_reference_offline,
)
from preprocessing.validator import print_report, load_csv, save_csv
import pandas as pd

## 1. Single Hansard Processing

In [ ]:
# Path to a sample Hansard text file
SAMPLE_TXT = "../data/hansards/txt/Hansard_1st_August_2025.txt"

# Check if file exists
if not Path(SAMPLE_TXT).exists():
    print(f"Sample file not found: {SAMPLE_TXT}")
    print("Please update SAMPLE_TXT to point to an existing file")
else:
    print(f"Processing: {SAMPLE_TXT}")

In [ ]:
# Step 1: Parse the Hansard
parser = HansardParser(SAMPLE_TXT)

# Print summary
parser.print_summary()

# Convert to DataFrame for exploration
import pandas as pd
df = pd.DataFrame([vars(s) for s in parser.statements])

print(f"\nDataFrame shape: {df.shape}")
df.head()

In [ ]:
# Save parsed output
parsed_path = "sample_parsed.csv"
parser.to_csv(parsed_path)
print(f"Saved to: {parsed_path}")

## 2. Validation (T1-T10)

In [ ]:
# Run validation suite
results = validate_hansard(parsed_path, SAMPLE_TXT, apply_fix=True)

# Print report
print_report(results, parsed_path, SAMPLE_TXT, apply_fix=True)

In [ ]:
# Save validated version
validated_path = "sample_validated.csv"
rows = load_csv(parsed_path)
save_csv(rows, validated_path)
print(f"Validated data saved to: {validated_path}")

## 3. Enrichment with MP Reference

In [ ]:
# Build or load MP reference
# For demo, use offline sample
ref_records = build_reference_offline()

import csv
ref_path = "sample_mp_reference.csv"
with open(ref_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=ref_records[0].keys())
    writer.writeheader()
    writer.writerows(ref_records)
    
print(f"MP reference: {len(ref_records)} MPs")
pd.DataFrame(ref_records).head()

In [ ]:
# Enrich Hansard with MP data
enriched, report, out_path = enrich_hansard(
    validated_path,
    ref_path,
    "sample_enriched.csv"
)

# Print enrichment report
from preprocessing.enrichment import print_enrichment_report
print_enrichment_report(report, enriched)

In [ ]:
# View enriched data
df_enriched = pd.DataFrame(enriched)

# Show speakers with party attribution
speakers = df_enriched[
    (df_enriched['is_stage_direction'] == '0') 
    & (df_enriched['party'] != '')
][['name', 'title', 'party', 'constituency', 'region']].drop_duplicates()

print(f"Speakers with party attribution: {len(speakers)}")
speakers.head(10)

## 4. Batch Processing

Process all Hansards in a directory:

In [ ]:
# Run batch processing script
!python ../scripts/preprocess_bulk.py --help

In [ ]:
# Example command (run in terminal):
# python scripts/preprocess_bulk.py \
#     --input-dir data/hansards/txt \
#     --output-dir data/hansards/processed \
#     --ref data/hansards/processed/ghana_mps_9th_parliament.csv

## 5. Build Full MP Reference (Online)

To build complete reference from parliament.gh and Wikipedia:

In [ ]:
# Requires internet and requests/beautifulsoup4
# from preprocessing.mp_reference import build_reference_online
# full_ref = build_reference_online("ghana_mps_9th_parliament.csv")

---

## Summary

This notebook demonstrated:
1. ✅ Parsing Hansard text into structured speaker turns
2. ✅ Running T1-T10 validation suite
3. ✅ Enriching with MP party/constituency data
4. ✅ Batch processing all files

**Next Steps**:
- Run batch processing on full 2025-2026 corpus
- Validate party attribution rate ≥ 90%
- Proceed to LLM scoring pipeline